# 02 — Function Tools: Giving Agents Superpowers

**What you'll learn:**
- How to define Python functions as tools an agent can call
- Using type annotations and `@tool` for rich descriptions
- Giving an agent multiple tools
- How the function-calling loop works under the hood

---

## Why Tools?

An LLM on its own can only generate text. **Tools** let your agent reach out
to the real world — query databases, call APIs, run calculations.

The flow is:
1. User asks a question
2. Agent decides which tool(s) to call and with what arguments
3. Framework executes the tool and sends results back to the agent
4. Agent formulates a natural-language answer using the tool results

The agent **never sees your tool code** — it only sees the function name,
description, and parameter schema.

In [19]:
import os
from typing import Annotated

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import Field

from agent_framework import tool
from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

## Defining Tools

A tool is just a Python function. Use `Annotated[type, Field(description=...)]`
so the agent knows what each parameter means.

Let's build three tools for an insurance quoting assistant.

In [20]:
@tool(name="get_premium_quote", description="Get a premium quote for a life insurance policy.", approval_mode="never_require")
def get_premium_quote(
    age: Annotated[int, Field(description="Applicant's age in years")],
    coverage_type: Annotated[str, Field(description="Type of coverage: 'term' or 'whole'")],
    coverage_amount: Annotated[int, Field(description="Coverage amount in USD")],
) -> str:
    """Calculate a mock premium quote based on age, coverage type, and amount."""
    # Simple mock pricing logic
    base_rate = 0.05 if coverage_type == "term" else 0.12
    age_factor = 1 + (age - 25) * 0.02 if age > 25 else 1.0
    monthly_premium = round((coverage_amount / 1000) * base_rate * age_factor, 2)
    return (
        f"Quote for {coverage_type} life insurance:\n"
        f"  Coverage: ${coverage_amount:,}\n"
        f"  Applicant age: {age}\n"
        f"  Estimated monthly premium: ${monthly_premium:,.2f}"
    )


print("Tool defined: get_premium_quote")

Tool defined: get_premium_quote


In [21]:
@tool(approval_mode="never_require")
def check_policy_status(
    policy_number: Annotated[str, Field(description="The policy number to look up, e.g. 'POL-12345'")],
) -> str:
    """Check the current status of an insurance policy."""
    # Mock policy database
    policies = {
        "POL-10042": {"status": "Active", "holder": "Sarah Johnson", "type": "Term Life", "premium": 85.00},
        "POL-20187": {"status": "Lapsed", "holder": "Mike Chen", "type": "Whole Life", "premium": 210.00},
        "POL-30291": {"status": "Pending Review", "holder": "Lisa Park", "type": "Term Life", "premium": 62.00},
    }
    policy = policies.get(policy_number)
    if not policy:
        return f"Policy {policy_number} not found in the system."
    return (
        f"Policy {policy_number}:\n"
        f"  Holder: {policy['holder']}\n"
        f"  Type: {policy['type']}\n"
        f"  Status: {policy['status']}\n"
        f"  Monthly Premium: ${policy['premium']:.2f}"
    )


print("Tool defined: check_policy_status")

Tool defined: check_policy_status


In [22]:
@tool(approval_mode="never_require")
def get_claim_history(
    customer_id: Annotated[str, Field(description="The customer ID to look up claims for")],
) -> str:
    """Retrieve the claim history for a customer."""
    # Mock claims database
    claims = {
        "CUST-001": [
            {"claim_id": "CLM-5001", "date": "2025-03-15", "type": "Auto", "amount": 4200, "status": "Paid"},
            {"claim_id": "CLM-5042", "date": "2025-11-02", "type": "Home", "amount": 12500, "status": "Under Review"},
        ],
        "CUST-002": [
            {"claim_id": "CLM-6010", "date": "2024-07-20", "type": "Auto", "amount": 1800, "status": "Paid"},
        ],
    }
    customer_claims = claims.get(customer_id)
    if not customer_claims:
        return f"No claims found for customer {customer_id}."
    lines = [f"Claims for {customer_id}:"]
    for c in customer_claims:
        lines.append(f"  - {c['claim_id']} ({c['date']}): {c['type']} — ${c['amount']:,} — {c['status']}")
    return "\n".join(lines)


print("Tool defined: get_claim_history")

Tool defined: get_claim_history


## Creating an Agent with Tools

Pass your tools to the agent via the `tools=` parameter.
The agent will automatically decide which tool to call based on the user's question.

In [23]:
agent = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
).as_agent(
    name="InsuranceAssistant",
    instructions=(
        "You are a helpful insurance assistant. "
        "Use the available tools to look up policy information, provide premium quotes, "
        "and retrieve claim history. Always be accurate and cite the data from tools."
    ),
    tools=[get_premium_quote, check_policy_status, get_claim_history],
)

print(f"Agent '{agent.name}' created with {len([get_premium_quote, check_policy_status, get_claim_history])} tools")

Agent 'InsuranceAssistant' created with 3 tools


## Demo 1: Getting a Premium Quote

The agent will figure out it needs to call `get_premium_quote` and extract
the right parameters from the natural-language question.

In [24]:
result = await agent.run(
    "I'm 35 years old. How much would a $500,000 term life policy cost me per month?"
)
print(result)

For a $500,000 term life insurance policy, a 35-year-old applicant would pay an estimated monthly premium of **$30.00**.


## Demo 2: Checking a Policy

Now the agent calls `check_policy_status` with the policy number.

In [25]:
result = await agent.run("Can you check the status of policy POL-20187?")
print(result)

The status of policy **POL-20187** is as follows:

- **Holder**: Mike Chen  
- **Type**: Whole Life  
- **Status**: Lapsed  
- **Monthly Premium**: $210.00  

Let me know if you'd like assistance with reactivating the policy!


## Demo 3: Multiple Tool Calls

The agent may call **multiple tools** in a single run if the question
requires it.

In [26]:
result = await agent.run(
    "Look up the claim history for customer CUST-001 and also check "
    "the status of their policy POL-10042."
)
print(result)

Here’s the information for customer and policy:

### Claim History for Customer ID **CUST-001**:
1. **Claim ID:** CLM-5001  
   - **Date:** March 15, 2025  
   - **Type:** Auto Insurance  
   - **Amount:** $4,200  
   - **Status:** Paid  

2. **Claim ID:** CLM-5042  
   - **Date:** November 2, 2025  
   - **Type:** Home Insurance  
   - **Amount:** $12,500  
   - **Status:** Under Review  

---

### Policy Status for Policy Number **POL-10042**:
- **Holder:** Sarah Johnson  
- **Type:** Term Life Insurance  
- **Status:** Active  
- **Monthly Premium:** $85.00


## How It Works Under the Hood

```
User: "How much would a $500K term life policy cost for a 35-year-old?"
         │
         ▼
┌─────────────────────────────┐
│   Agent sends prompt + tool │
│   schemas to the LLM       │
└────────────┬────────────────┘
             │
             ▼
┌─────────────────────────────┐
│   LLM responds with a      │
│   tool_call request:        │
│   get_premium_quote(        │
│     age=35,                 │
│     coverage_type="term",   │
│     coverage_amount=500000  │
│   )                         │
└────────────┬────────────────┘
             │
             ▼
┌─────────────────────────────┐
│   Framework executes the    │
│   Python function locally   │
│   and sends result back     │
└────────────┬────────────────┘
             │
             ▼
┌─────────────────────────────┐
│   LLM formulates final     │
│   natural-language answer   │
└─────────────────────────────┘
```

The agent may loop through this cycle multiple times if it needs to call
several tools before formulating the final answer.

In [27]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- Any Python function can become a tool — just use type annotations
- The `@tool` decorator lets you customize name, description, and approval mode
- Pass tools via `tools=[fn1, fn2, ...]` when creating the agent
- The agent autonomously decides **which** tools to call and **when**
- Multiple tools can be called in a single run

**Next up:** [03 — Structured Output](./03-structured-output.ipynb) — get typed
Pydantic objects instead of free text (claim reports, risk assessments).